## Set up imports and project root

In [ ]:
# %% [markdown]
# # Model Training
# Train predictive models for FIFA 2026 match outcomes using engineered features.

# %%
import os
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib

# Set project root
os.chdir(r"c:\Github\fifa2026-prediction")
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## Load the engineered features

In [ ]:
# %% [markdown]
# ## Load Features

# %%
features_path = PROJECT_ROOT / "data" / "processed" / "features.csv"
df = pd.read_csv(features_path)

print(f"✅ Loaded features: {df.shape}")
print(df.head())


## Define features and target

In [5]:
# %% [markdown]
# ## Prepare Data

# %%
# Features we'll use
feature_cols = [
    "home_elo",
    "away_elo",
    "elo_diff",
    "is_home_host",
    "is_away_host",
    "is_neutral",
    "is_friendly",
]

# Create missing binary features
df['is_home_host'] = (df['home_team'] == df['country']).astype(int)
df['is_away_host'] = (df['away_team'] == df['country']).astype(int)
df['is_neutral'] = df['neutral'].astype(int)
df['is_friendly'] = (df['tournament'] == 'Friendly').astype(int)

X = df[feature_cols]

# Target: 1 if home team wins, 0 otherwise
y = (df["home_goals"] > df["away_goals"]).astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Class balance:", y.value_counts())


Feature matrix shape: (47399, 7)
Target shape: (47399,)
Class balance: 0    24160
1    23239
Name: count, dtype: int64


## Train/test split and scaling

In [6]:
# %% [markdown]
# ## Train/Test Split and Scaling

# %%
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Scaling complete")


Train shape: (37919, 7)
Test shape: (9480, 7)
✅ Scaling complete


## Train Logistic Regression

In [7]:
# %% [markdown]
# ## Logistic Regression

# %%
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("Classification Report:\n", classification_report(y_test, y_pred_lr))


Logistic Regression Accuracy: 0.5327004219409283
Confusion Matrix:
 [[2965 1928]
 [2502 2085]]
Classification Report:
               precision    recall  f1-score   support

           0       0.54      0.61      0.57      4893
           1       0.52      0.45      0.48      4587

    accuracy                           0.53      9480
   macro avg       0.53      0.53      0.53      9480
weighted avg       0.53      0.53      0.53      9480



## Save Logistic Regression model + scaler

In [8]:
# %% [markdown]
# ## Save Logistic Regression Model

# %%
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(exist_ok=True)

joblib.dump(log_reg, models_dir / "logistic_regression.pkl")
joblib.dump(scaler, models_dir / "scaler.pkl")

print("✅ Saved logistic_regression.pkl and scaler.pkl in models/")


✅ Saved logistic_regression.pkl and scaler.pkl in models/


## Train XGBoost model

In [9]:
# %% [markdown]
# ## XGBoost Classifier

# %%
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    subsample=0.9,
    colsample_bytree=0.9,
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))


XGBoost Accuracy: 0.5578059071729958
Confusion Matrix:
 [[3098 1795]
 [2397 2190]]
Classification Report:
               precision    recall  f1-score   support

           0       0.56      0.63      0.60      4893
           1       0.55      0.48      0.51      4587

    accuracy                           0.56      9480
   macro avg       0.56      0.56      0.55      9480
weighted avg       0.56      0.56      0.56      9480



## Save XGBoost model

In [10]:
# %% [markdown]
# ## Save XGBoost Model

# %%
joblib.dump(xgb, models_dir / "xgboost.pkl")
print("✅ Saved xgboost.pkl in models/")


✅ Saved xgboost.pkl in models/
